# Vae model selection

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import glob
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Custom Functions

In [52]:
def loss_plot(
    models_dict,
    n_epochs_thr,
    save_to,
    figsize=(20, 10)
):

    fig, axs = plt.subplots(
        2, 1,
        figsize=figsize
    )

    for model_name, model in models_dict.items():


        if len(model.history['loss']) <= n_epochs_thr:
            continue

        train_loss = model.history['loss']
        val_loss = model.history['val_loss']
        epochs = np.arange(0, len(train_loss))

        val_loss_thr = model.history['val_loss'][n_epochs_thr:]
        train_loss_thr = train_loss[n_epochs_thr:]
        epochs_thr = np.arange(n_epochs_thr, len(train_loss_thr) + n_epochs_thr)


        print(f'Model {model_name}', end='\r')

        axs[0].clear()
        axs[1].clear()

        # Loss
        axs[0].plot(
            epochs, train_loss,
            color="black", label="Training Loss"
        )

        axs[0].plot(
            epochs, val_loss,
            color="C1", label="Validation Loss"
        )

        axs[0].set_yscale("log")

        # Loss
        axs[1].plot(
            epochs_thr, train_loss_thr,
            color="black", label="Training Loss"
        )

        axs[1].plot(
            epochs_thr, val_loss_thr,
            color="C1", label="Validation Loss"
        )
        # Legends
        axs[0].legend(
            loc='upper left',
            frameon=False,
        )

        axs[1].legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/loss_{model_name}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [3]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
bin_ids = [f"bin_{i:02d}" for i in range(4)]

## Data

In [ ]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1
n_wave = wave.shape

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)


# bin_02

In [16]:
models_02_dict = {}

models_paths_02 = sorted(
    glob.glob(f"{models_dir}/bin_02/00*")
)

for path in models_paths_02:

    model_name = path.split("/")[-1]
    print(f"Loading model: {model_name}", end='\r')
    
    model = AutoEncoder(reload=True, reload_from=path)
    models_02_dict[model_name] = model

In [53]:
save_to = f"{models_dir}/bin_02/losses"
os.makedirs(save_to, exist_ok=True)

models_dict = models_02_dict
n_epochs_thr = 50

loss_plot(
    models_dict,
    n_epochs_thr,
    save_to,
    figsize=(20, 10)
)

## Winner
* 0021

# bin_01

In [54]:
models_01_dict = {}

models_paths_01 = sorted(
    glob.glob(f"{models_dir}/bin_01/00*")
)

for path in models_paths_01:

    model_name = path.split("/")[-1]
    print(f"Loading model: {model_name}", end='\r')
    
    model = AutoEncoder(reload=True, reload_from=path)
    models_01_dict[model_name] = model

In [55]:
save_to = f"{models_dir}/bin_01/losses"
os.makedirs(save_to, exist_ok=True)

models_dict = models_01_dict
n_epochs_thr = 50

loss_plot(
    models_dict,
    n_epochs_thr,
    save_to,
    figsize=(20, 10)
)

## Winner
* 0057
* 0063 (prefer this)

# bin_00

In [56]:
models_00_dict = {}

models_paths_00 = sorted(
    glob.glob(f"{models_dir}/bin_00/00*")
)

for path in models_paths_00:

    model_name = path.split("/")[-1]
    print(f"Loading model: {model_name}", end='\r')
    
    model = AutoEncoder(reload=True, reload_from=path)
    models_00_dict[model_name] = model

In [57]:
save_to = f"{models_dir}/bin_00/losses"
os.makedirs(save_to, exist_ok=True)

models_dict = models_00_dict
n_epochs_thr = 50

loss_plot(
    models_dict,
    n_epochs_thr,
    save_to,
    figsize=(20, 10)
)

## Winner
* 0051
